[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BradSegal/OxfordCDT2026ClinicalUtility/blob/main/extension/05_windows_and_unreadable_recordings.ipynb)

In [ ]:
# Setup: run this cell first. It installs the course helpers (about 20 seconds).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("clinical_utility") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/BradSegal/OxfordCDT2026ClinicalUtility.git"], check=True)
    importlib.invalidate_caches()

import plotly.io as pio

try:
    from google.colab import output
    output.enable_custom_widget_manager()
    pio.renderers.default = "colab"
except ImportError:
    pio.renderers.default = "plotly_mimetype"
print("Course helpers ready.")

# Window-level calls and unreadable recordings

Allow about 35 minutes. How should the calls of a classifier that scores individual
short windows be combined into a single decision for an entire recording to enter AF
review? And how should a recording that nobody can read be managed? In this section
you will codify rules for combining the classifier's calls on individual windows
into one decision, you will choose one such rule on one half of the recordings and
report it on the other, you will correct a reported sensitivity value for a
resolution mismatch, and you will price the different methods of managing an
unreadable recording.

The action is **send this recording for AF review**. This has the same cost
structure in reviews as the core notebook, i.e. $C_{TP}$ = 1, $C_{FP}$ = 1, $C_{FN}$
= 5 and $C_{TN}$ = 0. Please attempt to solve each problem on your own by replacing
each `...` and running the checker below it. If you get stuck for about 10 minutes,
use the stuck switch underneath the problem, where there is one, or open the worked
copy in `extension/solutions/`. A later cell that needs an unfinished problem will
print `Waiting for ...`. To resolve this, please complete or switch the problem,
then click in the waiting cell, choose **Runtime > Run before**, then run the
waiting cell.

The problem with the CinC data is that it provides a single label for an entire
recording and not the specific periods within the recording where AF is occurring.
This means that while it is possible to determine the impact of a window decision
rule on a recording basis, it is not possible to validate where specifically within
the recording the windows are calling AF, the AF intervals, or the AF burden. As
such we need to look towards a synthetic dataset that has known intervals of AF as
well as artefacts. This dataset consists of 400 recordings that follow a simplified
version of the recording lengths of the CinC dataset (which range from about nine
seconds to just over a minute). 70% of recordings are 30 seconds long and the
remaining are a maximum of 60 seconds long. A classifier produces predictions for
10-second windows sampled every 5 seconds and so each window overlaps the next by
half, with between 5 and 11 windows per recording.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.metrics import confusion_matrix, recall_score

from clinical_utility import data, deployment, reference, windows
from clinical_utility import extension_checks as check, extension_plots as view

costs = dict(reference.DEFAULT_COSTS)
recordings = windows.synthetic_window_recordings()
valid = recordings["valid"]
window_positive = (recordings["window_scores"] >= 0.5) & valid
truth = recordings["af_present"]

## Match the label to the question

The arrays of windows have a row per recording and are padded to the length of the
longest recording. Actual windows are indicated by `valid` with padding never being
positive. A window is considered to be positive if it has a score greater than or
equal to 0.5. The reference standard we have here is **any AF in the recording**.
This is useful for determining if a recording that contains AF would be reviewed;
however, it does not provide information as to when AF starts or stops, how much AF
is present, or if a particular window call was correct. This requires ground truth
at an interval level which we have for these recordings but do not for the CinC
data.

We can define several rules for collating individual window calls into a single
decision. We can have any positive window result in the recording being sent for
review (*any window*), we can require more than half of the recording's own windows
to be positive (*majority*), we can require at least k positive windows anywhere
(*at least k windows*), or we can require k consecutive positive windows (*a run of
k*). With this geometry, k consecutive windows cover `window + (k - 1) * stride`
seconds: if we require two consecutive windows, they cover 15 seconds, and if we
require three, it's 20. This is not the same as requiring that duration of
continuous AF, as each window is called individually and there is no guarantee that
AF fills the window.

### Problem A: Make

Write `aggregate(positive, valid, k)` for boolean arrays shaped
`(recordings, windows)`. Return a dict of boolean arrays at keys `any`, `majority`
and `count` with one decision per recording. Majority is strictly more than half of
`valid.sum(axis=1)` due to variable recording length; count includes equality at
`k`.

In [ ]:
def aggregate(positive, valid, k):
    return {"any": ..., "majority": ..., "count": ...}
_ = check.run("window_rules", aggregate)

In [ ]:
# Stuck for about 10 minutes? Delete the # on the next line and run this cell.
# aggregate = check.use_reference("aggregate")

### Problem B: Make

Write `persistent(positive, run_length)`, where True indicates at least `run_length`
consecutive positive windows within a given recording.
`sliding_window_view(positive, run_length, axis=1)` produces every block of
`run_length` neighbouring windows. A recording qualifies if any such block is all
positive. Summation across all windows cannot distinguish between a contiguous run
and separated artefacts.

In [ ]:
def persistent(positive, run_length):
    return ...
_ = check.run("window_persistence", persistent)

In [ ]:
# Stuck for about 10 minutes? Delete the # on the next line and run this cell.
# persistent = check.use_reference("persistent")

In [ ]:
# An artefact recording and one with a brief AF episode, against the reference AF windows.
for kind in ("artefact", "paroxysmal"):
    i = np.flatnonzero(recordings["kind"] == kind)[0]
    n_windows = valid[i].sum()
    display(view.windows(recordings["recording"][i], recordings["window_scores"][i, :n_windows],
                         recordings["af_windows"][i, :n_windows], 0.5))
    print(kind, "AF interval in seconds [start, stop):", recordings["af_interval"][i])

## Choose a rule on one half, report it on the other

The recordings are divided into a *choose* half and a *report* half. We use the
choose half to select the rule based on the priced loss (breaking ties by the one
that requires the fewest reviews, and then by the earlier rule in the list). We then
report this rule on the report half, irrespective of performance.

### Problem C: Make

Write `rule_table(decisions, af_present, kind, costs, per=1000)`. `decisions` is a
dictionary with the rule name as a key and the decision array as the value. The
function should produce a DataFrame with the rule name as the index, and columns for
`reviews`, `af_detected`, `af_missed`, `artefact_reviewed` (number of kind
`"artefact"` recordings reviewed) and `brief_af_missed` (number of kind
`"paroxysmal"` recordings not reviewed), all per `per` recordings, and `loss` (the
priced total, from `confusion_matrix(..., labels=[False, True])`).

In [ ]:
def rule_table(decisions, af_present, kind, costs, per=1000):
    return ...
_ = check.run("window_table", rule_table)

In [ ]:
# Stuck for about 10 minutes? Delete the # on the next line and run this cell.
# rule_table = check.use_reference("rule_table")

In [ ]:
if check.ready(aggregate=aggregate, persistent=persistent, rule_table=rule_table):
    basic, three = aggregate(window_positive, valid, 2), aggregate(window_positive, valid, 3)
    rules = {"any window": basic["any"], "majority": basic["majority"],
             "at least 2 windows": basic["count"], "at least 3 windows": three["count"],
             f"run of 2 ({windows.covered_span(2)} s)": persistent(window_positive, 2),
             f"run of 3 ({windows.covered_span(3)} s)": persistent(window_positive, 3)}
    tables = {}
    for half in ("choose", "report"):
        mask = recordings["half"] == half
        tables[half] = rule_table({k: v[mask] for k, v in rules.items()}, truth[mask],
                                  recordings["kind"][mask], costs)
    display(tables["choose"].round(1))
    chosen_rule = windows.choose_rule(tables["choose"])
    print("Chosen on the choose half:", chosen_rule)
    display(tables["report"].loc[[chosen_rule]].round(1))

Read the choose table by loss. Given these prices, *any window* is the cheapest
rule. Given that 79 of the recordings have a brief episode, and that 49 of these
have two or more positive windows and 75 have at least one, we can see that
requiring a second window for a positive result will miss many more of them. Given
that a miss is priced at 5 reviews, these extra misses cost more than the reviews
such a rule saves. On the report half, the chosen rule reviews 598 recordings per
1,000, misses 10.1 AF per 1,000, and reviews 206 artefact recordings per 1,000:
essentially every one. This effectively means that at this price you will review the
artefact recordings in order to catch the brief AF. A service that priced misses
lower, or with window scores less prone to artefact, could make a different choice.
The result depends on the window cut, the geometry and the prices, all of which are
stated.

### Problem D: Fix the bug

A report gives "recording sensitivity" using the function below. Please fix this
function in place so that it applies `rule` to generate one decision per recording
and returns, with `recall_score`, the fraction of recordings with AF that are sent
for review.

In [ ]:
def recording_sensitivity(window_positive, af_present, rule):
    """Fraction of AF recordings that the rule sends for review."""
    af_windows = np.repeat(np.asarray(af_present)[:, None], np.shape(window_positive)[1], axis=1)
    return recall_score(af_windows.ravel(), np.asarray(window_positive).ravel())


def any_window(positive):
    return np.asarray(positive, dtype=bool).any(axis=1)


print("Recording sensitivity of any window:")
_ = check.show(recording_sensitivity, window_positive, truth, any_window)
_ = check.run("window_sensitivity", recording_sensitivity)

<details><summary>Open after your own attempt</summary>

#### Window level is not recording level

The original code was copying each recording's label across all windows and then
scoring the windows. For recordings with only a brief episode of AF, the majority of
windows contain no AF, which results in a low sensitivity that is not representative
of either "how many AF recordings were reviewed" or "how many AF windows were
found". Window-level sensitivity needs window truth; recording-level sensitivity
needs one decision per recording.

## Recordings that nobody can read

An unreadable recording has no binary AF reference, as it is labelled in the CinC
dataset as noise and not as AF-negative. It also cannot be reviewed in the ordinary
sense. There are four options that the service can pursue to address this:

| Pathway | What happens | Undetected AF |
|---|---|---|
| Accept | No further action. | All of it. |
| Over-read | A clinician reads the noisy trace (one review). It is conclusive with probability `over_read_success`, and then AF is established. | When the read is inconclusive. |
| Repeat | Ask for a repeat recording, paying `repeat_cost` and `delay_cost` once. With probability `repeat_readable` the classifier then decides with its sensitivity and specificity, and a flagged repeat is reviewed. An unreadable repeat is over-read. | Classifier misses, and inconclusive over-reads. |
| Refer | Send for an alternative assessment, such as a clinic ECG, costing `refer_cost`. AF is then *referred for assessment*, not yet established. | None, by assumption. |

The AF probability `p` among unreadable recordings is unknown here; this is an
assumption and should be varied as such, not a measurement. The other inputs are
stated assumptions: a 50% success rate of over-reading the recording, a 60%
probability that the repeat recording will be readable, a cost of 0.2 reviews to
repeat the recording and a delay cost of 0.1 reviews. The cost of referral is a flat
3 reviews. The sensitivity and specificity for readable repeats are the forest's
development sensitivity and specificity. This assumes that the repeat recording will
have the same AF status and that readability is independent of AF.

### Problem E: Make

Write `unreadable_losses(p, costs, assumptions)` to calculate and return the
expected loss per unreadable recording for each of the four pathways, `accept`,
`over_read`, `repeat` and `refer`, as well as the cheapest pathway under `best`.
Note that the cost of a review is `costs["TP"]` if it finds AF and `costs["FP"]` if
it does not. The cost of undetected AF is `costs["FN"]` in addition to the cost of
any review it used.

</details>

In [ ]:
assumptions = {"over_read_success": 0.5, "repeat_readable": 0.6, "repeat_cost": 0.2,
               "delay_cost": 0.1, "refer_cost": 3.0}
scores = data.load_scores()
development = scores[scores["split"] == "development"]
best = reference.cheapest(reference.operating_points(development["af"], development["random_forest"], costs))
assumptions.update(sensitivity=best["sensitivity"], specificity=best["specificity"])

In [ ]:
def unreadable_losses(p, costs, assumptions):
    return {"accept": ..., "over_read": ..., "repeat": ..., "refer": ..., "best": ...}
_ = check.run("window_unreadable", unreadable_losses)

In [ ]:
# Stuck for about 10 minutes? Delete the # on the next line and run this cell.
# unreadable_losses = check.use_reference("unreadable_losses")

In [ ]:
if check.ready(unreadable_losses=unreadable_losses):
    for p in (0.1, 0.3):
        print(f"Assumed AF probability {p}:", {k: (round(v, 3) if not isinstance(v, str) else v)
                                               for k, v in unreadable_losses(p, costs, assumptions).items()})
        display(deployment.unreadable_consequences(p, assumptions).round(1))
    grid = [k / 100 for k in range(0, 51, 5)]
    print("Cheapest pathway by assumed AF probability:",
          {p: unreadable_losses(p, costs, assumptions)["best"] for p in grid})

Read the consequences per 1,000 unreadable recordings. If the assumed AF probability
is 0.1, the cheapest pathway is to simply accept the recording, which leaves 100 AF
recordings per 1,000 undetected. If we were to repeat the recording instead, we
would establish 65.7 of these at the cost of 470 reviews and a repeat for every
recording. The cheapest pathway will be to repeat recordings from an assumed AF
probability of about 0.26. Referral is never cheapest at a cost of 3 reviews here;
however, it is the only pathway that leaves no AF undetected, and what it produces
is not AF established, but rather AF referred for assessment. The cheapest pathway
is based on the AF probability among unreadable recordings, which would need to be
measured by the service, as well as on whether unreadable recordings are more common
during AF.

## Check your report

- What is the difference between the recording label, interval truth, and AF burden?
- How can you define a persistence rule as a covered span in seconds?
- Which half was used to select the persistence rule, and which to report it?
- Which assumption has the most impact on the unreadable pathway?

**Question to consider:** if a recording is more likely to be unreadable during AF,
how would you modify the pathways, and what data would you need to determine their
probabilities?